# 08.02 ATC 转换命令与参数

本分册围绕 ATC 命令展开：确定模型与目标芯片、填写输入 shape、选择精度策略，再检查动态 shape 配置。示例继续使用 `l08_workspace` 中的模型文件。

## 实验原理

ATC 通过命令行参数读取模型、指定目标芯片、设置输入 shape 和精度策略。参数形式统一为 `--参数=值`：

```bash
atc --model=xxx.onnx --framework=5 --output=xxx --soc_version=xxx
```

| 组别 | 代表参数 | 作用 |
| --- | --- | --- |
| 输入 | `--model` `--framework` `--input_shape` `--input_format` | 说明输入模型和张量 |
| 输出 | `--output` `--output_type` | 指定产物位置和类型 |
| 目标 | `--soc_version` | 指定编译目标芯片 |
| 精度 | `--precision_mode` `--precision_mode_v2` | 控制精度策略 |
| 动态 shape | `--dynamic_batch_size` `--dynamic_image_size` `--dynamic_dims` `--input_shape_range` | 声明可变输入 |
| 调试 | `--log` `--debug_dir` | 输出转换日志 |

参数名称和取值随 CANN 版本变化，执行前以本机 `atc --help` 为准。

In [ ]:
!source /usr/local/Ascend/ascend-toolkit/set_env.sh && atc --help | grep -c '^\s*--'

## 实验流程

### 1. 确定必填参数

#### `--model`

`--model` 指向待转换模型。模型文件由 `--framework` 决定：ONNX 使用 `.onnx`，TensorFlow 使用 `.pb`；Caffe 的结构和权重分离，`--model` 使用 `.prototxt`，并用 `--weight` 指定 `.caffemodel`。

相对路径以执行命令时的工作目录为准；带空格的路径需要加引号。

#### `--framework`

`--framework` 决定 ATC 使用的模型解析器。

| 值 | 框架 | 典型文件 |
| --- | --- | --- |
| 0 | Caffe | `.prototxt` + `.caffemodel` |
| 1 | MindSpore | `.air` / MindIR |
| 3 | TensorFlow | `.pb` |
| 5 | ONNX | `.onnx` |

本 Lab 使用 ONNX，对应值为 5。解析器与模型文件不匹配时，转换日志通常在解析阶段报错。


#### `--output`

`--output` 是输出路径前缀，不带扩展名；ATC 自动追加 `.om`。

```bash
--output=model         # 得到 model.om
--output=model.om      # 得到 model.om.om
```

输出目录需要预先创建。为不同转换版本命名时，可在前缀中记录 batch 或精度模式。


#### `--input_shape`

该参数指定输入节点名称和形状，格式为 `"名称:维度,维度,..."`。多输入用分号分隔：

```bash
--input_shape="input_ids:1,128;attention_mask:1,128;token_type_ids:1,128"
```

整个值使用双引号，避免 shell 将分号视为命令分隔符。输入名称与 ONNX 的 `input_names` 完全一致；可从模型中读取名称。

In [ ]:
import onnx

m = onnx.load("l08_workspace/demo_model.onnx")

parts = []
for i in m.graph.input:
    dims = [d.dim_value if d.dim_value else -1 for d in i.type.tensor_type.shape.dim]
    parts.append("%s:%s" % (i.name, ",".join(str(d) for d in dims)))

print('--input_shape="%s"' % ";".join(parts))

输出中的 `-1` 表示该维在 ONNX 中是动态维，需要与动态 shape 参数配合使用。

#### `--soc_version`

`--soc_version` 指定编译目标芯片型号。该值属于编译期配置，应通过 `npu-smi info` 或 `acl.get_soc_name()` 获取。


OM 面向特定芯片型号编译，部署到不同型号时需要重新转换。


### 2. 选择常用可选参数

#### `--input_format`

可用值包括 `NCHW`、`NHWC` 和 `ND`。PyTorch 导出的视觉模型通常使用 NCHW；NLP 模型和非图像张量使用 ND。它应与 `--input_shape` 的维度顺序一致。

#### `--log`

可用级别为 `debug`、`info`、`warning` 和 `error`。排查转换问题时，可使用 debug 日志。

#### `--precision_mode` 与 `--precision_mode_v2`

这两个参数控制全局精度策略。`--precision_mode` 的常见取值包括 `force_fp16`、`allow_fp32_to_fp16`、`must_keep_origin_dtype`、`allow_mixed_precision` 和 `force_fp32`；`--precision_mode_v2` 是较新的参数。

两者不能同时配置。默认值依赖 CANN 版本和芯片型号，可用本机 `atc --help` 核对。分析精度差异时，可额外转换一份 `must_keep_origin_dtype` 模型作对照。

In [ ]:
%%bash
source /usr/local/Ascend/ascend-toolkit/set_env.sh
export TE_PARALLEL_COMPILER=1
export MAX_COMPILE_PROCESS_NUM=1
atc --model=l08_workspace/demo_model.onnx \
    --framework=5 \
    --output=l08_workspace/demo_model_fp32 \
    --input_format=NCHW \
    --input_shape="input:1,3,32,32" \
    --soc_version=Ascend910B1 \
    --precision_mode=must_keep_origin_dtype

使用同一份输入运行两份 OM 并比较输出。保持原始精度的版本正常、默认版本异常时，检查精度模式；两份结果都异常时，回到模型导出和转换参数。

### 3. 设置动态 shape

当 batch、图像分辨率或序列长度变化时，在 ATC 命令中声明可支持的 shape。

| 模式 | ATC 参数 | `--input_shape` 写法 | 适用 |
| --- | --- | --- | --- |
| 静态 | 无 | 全部写实际值 | 输入固定 |
| 动态 batch | `--dynamic_batch_size="1,2,4,8"` | batch 维写 `-1` | 只有 batch 变化 |
| 动态分辨率 | `--dynamic_image_size="224,224;448,448"` | H/W 维写 `-1` | 图像尺寸变化 |
| 动态维度 | `--dynamic_dims="16;32;64"` | 对应维写 `-1` | 指定维度变化 |
| 完全动态 | `--input_shape_range="input:[-1,3,-1,-1]"` | 使用 range 语法 | shape 完全不定 |

使用动态参数时，可变维在 `--input_shape` 中写为 `-1`。三个 `--dynamic_*` 参数互斥；`--dynamic_dims` 还要与 `--input_format` 配合。档位之间用分号分隔，档位内部用逗号。

In [ ]:
import torch
import torch.nn as nn

# 创建工作目录
import os
os.makedirs("l08_workspace", exist_ok=True)


# 一个简单 CNN 模型
class DemoModel(nn.Module):
    def __init__(self):
        super().__init__()
        self.net = nn.Sequential(
            nn.Conv2d(3, 16, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.Conv2d(16, 3, kernel_size=3, padding=1),
            nn.AdaptiveAvgPool2d((1, 1))
        )

    def forward(self, x):
        return self.net(x)


model = DemoModel()
model.eval()


# dummy input
dummy_input = torch.randn(1, 3, 32, 32)


# 导出动态 batch ONNX
torch.onnx.export(
    model,
    dummy_input,
    "l08_workspace/demo_model_dyn.onnx",
    opset_version=11,
    input_names=["input"],
    output_names=["output"],
    dynamic_axes={
        "input": {
            0: "batch"
        },
        "output": {
            0: "batch"
        }
    }
)

print("ONNX export success")

In [ ]:
%%bash
source /usr/local/Ascend/ascend-toolkit/set_env.sh
export TE_PARALLEL_COMPILER=1
export MAX_COMPILE_PROCESS_NUM=1
atc --model=l08_workspace/demo_model_dyn.onnx \
    --framework=5 \
    --output=l08_workspace/demo_model_dyn \
    --input_format=NCHW \
    --input_shape="input:-1,3,32,32" \
    --dynamic_batch_size="1,2,4,8" \
    --soc_version=Ascend910B1

每个档位都会生成对应的执行调度信息。档位数量增加时，转换时间和 OM 体积也会增加；按实际请求的 shape 选择档位即可。


### 4. 检查转换命令

执行转换前，按下表核对参数关系。

| 检查项 | 常见错误 |
| --- | --- |
| 必填参数齐全 | 漏 `--soc_version` |
| `--output` 不带 `.om` | 写成 `model.om`，得到 `model.om.om` |
| 模型后缀与 framework 匹配 | `.onnx` 配 `--framework=3` |
| `--input_shape` 语法 | 多输入用了逗号；整个值未加引号 |
| 输入名与 ONNX 一致 | 手抄错字；模型重新导出后名称变化 |
| `-1` 与 `--dynamic_*` 配套 | 有 `-1` 却未给档位；给了档位却未写 `-1` |
| `--dynamic_*` 互斥 | 同时配置 batch 和 image_size |
| 精度模式互斥 | `--precision_mode` 与 `_v2` 同时出现 |
| `soc_version` 为实测值 | 照抄文档中的示例值 |
| 输出目录存在 | ATC 不会自动创建 |

In [ ]:
%%bash
source /usr/local/Ascend/ascend-toolkit/set_env.sh
export TE_PARALLEL_COMPILER=1
export MAX_COMPILE_PROCESS_NUM=1
atc --model=l08_workspace/demo_model.onnx \
    --framework=5 \
    --output=l08_workspace/demo_model_bs1_fp16 \
    --input_format=NCHW \
    --input_shape="input:1,3,32,32" \
    --soc_version=Ascend910B1\
    --log=info

echo "退出码: $?"
ls -lh l08_workspace/*.om

### 5. 阅读完整命令示例

#### CV 分类，静态 shape

```bash
atc --model=resnet50.onnx --framework=5 \
    --output=resnet50_bs1 \
    --input_format=NCHW --input_shape="input:1,3,224,224" \
    --soc_version=<实测值>
```

#### CV 检测，动态分辨率

```bash
atc --model=yolov5.onnx --framework=5 \
    --output=yolov5_dynres \
    --input_format=NCHW --input_shape="images:1,3,-1,-1" \
    --dynamic_image_size="640,640;1280,1280" \
    --soc_version=<实测值>
```

#### NLP 多输入，动态 batch

```bash
atc --model=bert.onnx --framework=5 \
    --output=bert_dynbs \
    --input_format=ND \
    --input_shape="input_ids:-1,128;attention_mask:-1,128;token_type_ids:-1,128" \
    --dynamic_batch_size="1,4,8,16" \
    --soc_version=<实测值>
```

第三个示例中，多输入使用分号分隔；非图像张量使用 ND；每个输入的 batch 维都声明为 `-1`。


## 实验总结

| 要回答的问题 | 对应参数 |
| --- | --- |
| 模型是什么 | `--model` `--framework` `--input_shape` `--input_format` |
| 运行在哪个芯片 | `--soc_version` |
| 精度与动态 shape 如何设置 | `--precision_mode` `--dynamic_*` |
| 产物写到哪里 | `--output` |

常用参数：

```text
必填  --model --framework --output --input_shape --soc_version
常用  --input_format --log --precision_mode
动态  --dynamic_batch_size | --dynamic_image_size | --dynamic_dims
      --input_shape_range
调试  --log=debug --debug_dir
```


## 实验扩展

1. 为一个三输入 NLP 模型写出正确的 `--input_shape`。输入名为 `input_ids`、`attention_mask`、`token_type_ids`，形状均为 `(1, 128)`。
2. 找出下面命令中的错误：
   ```bash
   atc --model=model.onnx --framework=3 --output=model.om \
       --input_shape="a:1,3,224,224,b:1,10" --soc_version=Ascend310
   ```
3. 将 08.01 的静态命令改为支持 1/2/4/8 四个 batch 档位，并重新导出 ONNX。
4. 用 `must_keep_origin_dtype` 和默认精度各转换一份 OM，比较输出差异并记录数值。
5. 在昇腾环境运行 `atc --help`，核对本分册的参数是否存在并记录差异。
6. 为什么档位增加会使 OM 变大、转换变慢？结合 OM 的组成说明。


## 参考答案


In [ ]:
!cat answer/L08-02_answer.txt